# Experiment Results Visualization
Loads pre-computed JSON/CSV result files and renders:
- Table 1: Main comparison (deep / RF / stack vs baselines)
- Table 2: Ablation study
- Figure 1: Regime membership radar
- Figure 2: Expert weight heatmap
- Figure 3: Case study (prediction vs ground-truth)
- Figure 4: Soft-gate entropy vs RMSE

In [ ]:
import sys
sys.path.insert(0, '..')

import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})
RESULTS = Path('../results')
INTERPRET = RESULTS / 'interpret'
STATIONS = [f'station{i:02d}' for i in range(10)]

## Table 1 — Main results (deep / RF / stack)

In [ ]:
def load_stack_results(results_dir=RESULTS):
    rows = []
    for p in sorted(results_dir.glob('stack_*.json')):
        with open(p) as f:
            rows.append(json.load(f))
    return rows

stack_rows = load_stack_results()
methods = ['deep', 'rf', 'stack']
table1_rows = []
for m in methods:
    vals = pd.DataFrame([r[m] for r in stack_rows])
    table1_rows.append({
        'Method': {'deep': 'MoE-Deep', 'rf': 'MoE-RF', 'stack': 'MoE-Stack'}[m],
        'ACC (%)': f"{vals['acc'].mean():.2f} ± {vals['acc'].std():.2f}",
        'RMSE': f"{vals['rmse'].mean():.4f} ± {vals['rmse'].std():.4f}",
        'MAE': f"{vals['mae'].mean():.4f} ± {vals['mae'].std():.4f}",
        'R²': f"{vals['r2'].mean():.4f} ± {vals['r2'].std():.4f}",
        'QR (%)': f"{vals['qr'].mean():.2f} ± {vals['qr'].std():.2f}",
    })
table1 = pd.DataFrame(table1_rows).set_index('Method')

print("""
【如何看这张表】
- 行：本文方法的三个输出层次
  · MoE-Deep  — 异构专家混合模型的深度学习直接输出
  · MoE-RF    — 用随机森林对深度输出做二次拟合
  · MoE-Stack — 用线性融合权重将 Deep 和 RF 结果 stacking（最终提交结果）
- 列：评估指标（10站×5随机种子均值±标准差）
  · ACC  = 1 - NMAE（越高越好，满分100）
  · RMSE / MAE — 绝对误差（越低越好）
  · R²   — 拟合优度（越高越好，满分1）
  · QR   — 分位数覆盖率（衡量预测置信区间，越高越好）
- 结论：Stack > RF > Deep，stacking 带来约+0.3% ACC 增益。
""")
print('=== Table 1: Main results (10 stations × 5 seeds) ===')
table1

In [ ]:
print("""
【如何看这张图】
- 每组3根柱子对应同一站点的 Deep/RF/Stack 三个层次
- 误差棒 = 5个随机种子的标准差，反映模型稳定性
- Stack（红色）几乎在所有站点都高于 Deep 和 RF
- 站点间 ACC 差异（约83-95%）反映各站气象复杂度的差异
""")

per_station = {m: {} for m in methods}
for r in stack_rows:
    sid = r['station']
    for m in methods:
        per_station[m].setdefault(sid, []).append(r[m]['acc'])

x = np.arange(len(STATIONS))
width = 0.25
colors = ['#4878CF', '#6ACC65', '#D65F5F']
labels = ['MoE-Deep', 'MoE-RF', 'MoE-Stack']

fig, ax = plt.subplots(figsize=(14, 5))
for i, (m, color, label) in enumerate(zip(methods, colors, labels)):
    means = [np.mean(per_station[m].get(s, [np.nan])) for s in STATIONS]
    stds  = [np.std(per_station[m].get(s, [np.nan]))  for s in STATIONS]
    ax.bar(x + i * width, means, width, yerr=stds, label=label,
           color=color, capsize=3, alpha=0.85)
ax.set_xticks(x + width)
ax.set_xticklabels([s[-2:] for s in STATIONS])
ax.set_xlabel('Station ID')
ax.set_ylabel('ACC (%)')
ax.set_title('Per-station ACC: Deep / RF / Stack (mean ± std over 5 seeds)')
ax.legend()
ax.set_ylim(80, 100)
plt.tight_layout()
plt.savefig('../results/fig_main_acc.pdf', bbox_inches='tight')
plt.show()

## Table 1b — Baseline comparison

In [ ]:
BASELINE_DESC = {
    'rf_hist':          'RF-Hist    — 随机森林，输入历史功率序列',
    'rf_nwp':           'RF-NWP     — 随机森林，输入NWP气象特征',
    'DLinear':          'DLinear    — 线性分解模型，将序列拆为趋势+余项分别线性映射（ICLR 2023）',
    'LSTM':             'LSTM       — 长短期记忆网络，经典RNN序列模型',
    'LSTNet':           'LSTNet     — CNN+LSTM+跳跃连接，针对多变量时序设计（KDD 2018）',
    'TCN':              'TCN        — 时序卷积网络，因果空洞卷积替代RNN（ICLR 2018）',
    'NBEATS':           'N-BEATS    — 纯MLP+残差块，无需特征工程（ICLR 2020）',
    'NHiTS':            'N-HiTS     — N-BEATS升级版，多尺度插值降低参数量（AAAI 2023）',
    'Crossformer':      'Crossformer— 跨维度Transformer，同时建模时间和变量间依赖（ICLR 2023）',
    'NWPLSTMbaseline':  'NWP-LSTM   — 以NWP预报直接作为输入的LSTM，代表业务场景基线',
    'Informer':         'Informer   — 稀疏自注意力Transformer，长序列预测先驱（AAAI 2021）',
    'PatchTST':         'PatchTST   — 将序列分patch后用Transformer，单变量独立建模（ICLR 2023）',
    'iTransformer':     'iTransformer—倒置Transformer，在变量维度做注意力（ICLR 2024）',
    'TimesNet':         'TimesNet   — 将1D时序变换为2D特征图用CNN处理（ICLR 2023）',
}

bl_csv = RESULTS / 'baselines_table.csv'
if bl_csv.exists():
    bl = pd.read_csv(bl_csv, index_col=0)
    print("""
【如何看这张表】
- 行：各baseline模型（见下方模型说明）
- 列：各站点的 ACC 均值±std（跨5种子），MEAN列为10站均值
- 数值越高越好（ACC = 1 - NMAE，满分100）
- 与 Table 1 的 MoE-Stack(90.51) 对比即可看到本文方法的提升幅度

【模型说明】""")
    for k, v in BASELINE_DESC.items():
        if k in bl.index:
            print(f'  {v}')
    print()
    display(bl)
else:
    print(f'{bl_csv} not found — run scripts/train_baselines.sh first')

## Table 2 — Ablation study

In [ ]:
abl_csv = RESULTS / 'ablation' / 'summary.csv'
if not abl_csv.exists():
    print(f'{abl_csv} not found — run scripts/ablation.sh first')
else:
    abl = pd.read_csv(abl_csv)
    VARIANT_LABEL = {
        'A':  'A  完整模型（Full）',
        'B':  'B  硬门控（Hard gating）',
        'C':  'C  均等固定门控（Fixed equal gate）',
        'D':  'D  同构专家（Homogeneous experts）',
        'E':  'E  单专家（Single expert）',
        'G2': 'G2 K=2 个天气模态',
        'G4': 'G4 K=4 个天气模态',
        'G5': 'G5 K=5 个天气模态',
        'H':  'H  VMD使用全量数据（oracle泄漏上界）',
        'I':  'I  辐照度锚点约束',
    }
    abl['label'] = abl['variant'].map(VARIANT_LABEL).fillna(abl['variant'])
    baseline_acc = float(abl[abl['variant'] == 'A']['acc_mean'].values[0])
    abl['Δ ACC'] = abl['acc_mean'] - baseline_acc

    print("""
【如何看这张表】
- 行：消融变体，A 为完整模型，其余每行移除或替换一个组件
  · B/C — 验证可学习软门控的作用：硬门控/固定门控 vs 软门控
  · D   — 验证异构专家的必要性：换成同构专家（相同结构）
  · E   — 验证多专家的必要性：退化为单个专家
  · G2/G4/G5 — K 的敏感性分析（K=3为完整模型）
  · H   — VMD在全数据上分解（信息泄漏），作为性能上界
  · I   — 增加辐照度锚点正则约束
- Δ ACC = 该变体 - 完整模型A，负值说明该组件有贡献
- 结论：移除任一核心组件均导致性能下降，K=3最优
""")
    display_cols = ['label', 'acc_mean', 'acc_std', 'rmse_mean', 'mae_mean', 'r2_mean', 'Δ ACC']
    out = abl[display_cols].rename(columns={
        'label': 'Variant', 'acc_mean': 'ACC mean', 'acc_std': 'ACC std',
        'rmse_mean': 'RMSE', 'mae_mean': 'MAE', 'r2_mean': 'R²'
    })
    display(out.style.format({
        'ACC mean': '{:.2f}', 'ACC std': '{:.2f}',
        'RMSE': '{:.4f}', 'MAE': '{:.4f}', 'R²': '{:.4f}', 'Δ ACC': '{:+.2f}'
    }))

## Figure 1 — 天气模态软隶属度分布
# 每个测试样本（96步预测窗口）对应一组 FCM 软隶属度 (u0, u1, u2)，
# 三角图中每个点代表一天，点越靠近某顶角说明该天越属于对应模态。
# 颜色深浅 = regime_0 的隶属度大小。

In [ ]:
sid = 'station00'
rm_path = INTERPRET / f'regime_membership_{sid}.csv'

if not rm_path.exists():
    print(f'{rm_path} not found — run: python -m src.run interpret --station {sid}')
else:
    rm = pd.read_csv(rm_path)
    regime_cols = [c for c in rm.columns if c.startswith('regime_')]
    K = len(regime_cols)
    U = rm[regime_cols].values  # (N_samples, K)

    print(f"""
【如何看这张图】
- 三角图（ternary plot）：三顶角分别代表三种天气模态
- 每个散点 = 一个预测窗口的软隶属度 (u0, u1, u2)，三者之和=1
- 点越靠近某顶角 → 该窗口越"纯粹"属于该模态
- 颜色越蓝 → regime_0 隶属度越高；越红 → 越低
- 点集中于顶角附近说明 FCM 区分度好；聚在中央说明模态边界模糊
""")

    if K == 3:
        def to_cart(u):
            s = u.sum(axis=1, keepdims=True)
            x = 0.5 * (2 * u[:, 1] + u[:, 2]) / s[:, 0]
            y = (np.sqrt(3) / 2) * u[:, 2] / s[:, 0]
            return x, y

        x, y = to_cart(U)

        fig, ax = plt.subplots(figsize=(7, 6.5))
        sc = ax.scatter(x, y, c=U[:, 0], cmap='RdYlBu',
                        s=6, alpha=0.35, vmin=0, vmax=1, zorder=2)
        plt.colorbar(sc, ax=ax, label='regime_0 membership', shrink=0.7)

        # 三角边框
        tri = plt.Polygon([[0, 0], [1, 0], [0.5, np.sqrt(3)/2]],
                           fill=False, edgecolor='k', lw=1.2, zorder=3)
        ax.add_patch(tri)

        # 顶角标签：放在三角形顶点外侧，留足偏移避免重叠
        offset = 0.10
        ax.text(0.5,  np.sqrt(3)/2 + offset, 'Regime 2\n(晴天型)',
                ha='center', va='bottom', fontsize=10, color='#2ca02c')
        ax.text(-offset * 1.2, -offset * 0.6, 'Regime 0\n(阴天型)',
                ha='center', va='top', fontsize=10, color='#1f77b4')
        ax.text(1 + offset * 1.2, -offset * 0.6, 'Regime 1\n(多云型)',
                ha='center', va='top', fontsize=10, color='#ff7f0e')

        ax.set_xlim(-0.25, 1.25)
        ax.set_ylim(-0.18, 1.05)
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(f'{sid} — FCM regime membership (K=3)', pad=12)
        plt.tight_layout()
        plt.savefig('../results/fig1_regime_ternary.pdf', bbox_inches='tight')
        plt.show()
    else:
        means = U.mean(axis=0)
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.bar(range(K), means, color='steelblue')
        ax.set_xlabel('Regime')
        ax.set_ylabel('Mean membership')
        ax.set_title(f'{sid} — Mean FCM regime membership')
        plt.tight_layout()
        plt.savefig('../results/fig1_regime_bar.pdf', bbox_inches='tight')
        plt.show()

## Figure 2 — Expert weight heatmap
Requires `expert_weight_{sid}.csv` from `src.run interpret`.

In [ ]:
sid = 'station00'
ew_path = INTERPRET / f'expert_weight_{sid}.csv'
ee_path = INTERPRET / f'entropy_error_{sid}.csv'

if not ew_path.exists():
    print(f'{ew_path} not found — run: python -m src.run interpret --station {sid}')
else:
    ew = pd.read_csv(ew_path)   # columns: expert_0, expert_1, expert_2
    ee = pd.read_csv(ee_path)   # columns: entropy, rmse_day, dominant_regime
    expert_cols = [c for c in ew.columns if c.startswith('expert_')]
    K = len(expert_cols)

    ew['dominant_regime'] = ee['dominant_regime'].values
    pivot = ew.groupby('dominant_regime')[expert_cols].mean()
    pivot.columns = [f'Expert {k}' for k in range(K)]
    pivot.index   = [f'Regime {r}' for r in pivot.index]

    print(f"""
【如何看这张图】
- 行：dominant regime（该样本软隶属度最高的模态，即"主导天气类型"）
- 列：各专家的平均门控权重（行内之和≈1）
- 颜色越深 → 该模态下对应专家被激活程度越高
- 理想情况：每个 regime 对应一个主力专家，形成对角线高亮
  → 说明不同天气类型确实由不同专家负责，模型具备可解释的模态专一性
- 若权重均匀分布则说明门控未学到有效分工
""")

    fig, ax = plt.subplots(figsize=(7, 4))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd',
                vmin=0, vmax=1, linewidths=0.5, ax=ax)
    ax.set_title(f'{sid} — Mean expert gate weights per dominant regime')
    plt.tight_layout()
    plt.savefig('../results/fig2_expert_heatmap.pdf', bbox_inches='tight')
    plt.show()

## Figure 3 — Case study: prediction vs ground-truth

In [ ]:
sid = 'station00'
seed = 0

# 找 predictions 文件（文件名含时间戳，用 glob 匹配）
pred_candidates = sorted((RESULTS / 'ablation' / 'A' / 'eval').glob(f'predictions_{sid}*.csv'))

if not pred_candidates:
    print(f'No predictions CSV found for {sid}. Run: python -m src.run evaluate --station {sid}')
else:
    pred = pd.read_csv(pred_candidates[seed])
    # columns: day_id, step, true_power, pred_power, abs_error, is_day
    print(f'Loaded: {pred_candidates[seed].name}  shape={pred.shape}')

    # 取3个完整的白天（跳过纯夜间）
    day_ids = pred[pred['is_day'] == 1]['day_id'].unique()[:3]
    subset = pred[pred['day_id'].isin(day_ids)].copy()
    t = np.arange(len(subset)) * 15 / 60  # 小时

    print("""
【如何看这张图】
- 蓝线：实测功率（ground truth）
- 红线：模型预测功率
- x轴：时间（小时），每24h为一天，共展示3天
- y轴：归一化功率（0~1，已除以装机容量）
- 垂直虚线：日界线
- 关注点：
  · 早晚坡爬升/下降是否跟得上（斜坡跟踪能力）
  · 阴天/云遮时预测是否平滑过渡
  · 峰值时刻预测偏差大小
""")

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(t, subset['true_power'].values, label='Ground truth', lw=1.5, color='#2c7bb6')
    ax.plot(t, subset['pred_power'].values, label='Prediction',   lw=1.5, color='#d7191c', alpha=0.85)
    ax.set_xlabel('Time (h)')
    ax.set_ylabel('Power (normalized)')
    ax.set_title(f'{sid} seed={seed} — 3-day prediction case study (variant A)')
    ax.legend()
    for d in range(1, 3):
        ax.axvline(d * 24, color='gray', lw=0.8, ls='--')
    plt.tight_layout()
    plt.savefig('../results/fig3_case_study.pdf', bbox_inches='tight')
    plt.show()

## Figure 4 — Soft-gate entropy vs RMSE
Requires `entropy_error_{sid}.csv` from `src.run interpret`.

In [ ]:
sid = 'station00'
ee_path = INTERPRET / f'entropy_error_{sid}.csv'

if not ee_path.exists():
    print(f'{ee_path} not found — run: python -m src.run interpret --station {sid}')
else:
    ee = pd.read_csv(ee_path)  # columns: entropy, rmse_day, dominant_regime（逐样本）

    # 按 dominant_regime 分组后按熵排序，每组取均值代表"典型难度"
    # 直接用逐样本散点（样本数~5000，适度降采样避免过密）
    sample = ee.sample(frac=0.3, random_state=42).copy()
    colors_map = {0: '#1f77b4', 1: '#ff7f0e', 2: '#2ca02c'}
    regime_name = {0: 'Regime 0', 1: 'Regime 1', 2: 'Regime 2'}

    print("""
【如何看这张图】
- x轴：软门控的香农熵（bits）
  · 熵低（→0）= 门控非常确定，某一专家权重接近1（硬决策）
  · 熵高（→log K）= 门控均匀，多个专家贡献相近（软过渡/模糊天气）
- y轴：该样本的 RMSE（预测误差）
- 颜色：该样本的主导天气模态
- 红色回归线：总体趋势
- 关键结论：
  · 负相关（r<0）说明门控越确定 → 误差越小，软门控设计有效
  · 熵高的样本集中于过渡天气（如多云），这类天气本身预测难度大
""")

    fig, ax = plt.subplots(figsize=(8, 5))
    for r, grp in sample.groupby('dominant_regime'):
        ax.scatter(grp['entropy'], grp['rmse_day'],
                   s=8, alpha=0.35, color=colors_map.get(r, 'gray'),
                   label=regime_name.get(r, f'Regime {r}'))

    coef = np.polyfit(ee['entropy'], ee['rmse_day'], 1)
    x_line = np.linspace(ee['entropy'].min(), ee['entropy'].max(), 100)
    ax.plot(x_line, np.polyval(coef, x_line), color='crimson', lw=2,
            label=f'Linear fit (r={ee["entropy"].corr(ee["rmse_day"]):.3f})')

    ax.set_xlabel('Gate entropy (bits)')
    ax.set_ylabel('Sample RMSE')
    ax.set_title(f'{sid} — Soft-gate entropy vs prediction error')
    ax.legend(markerscale=2)
    plt.tight_layout()
    plt.savefig('../results/fig4_entropy_rmse.pdf', bbox_inches='tight')
    plt.show()

## Appendix — Per-station detailed metrics

In [ ]:
per_sid_rows = []
for r in stack_rows:
    sid = r['station']
    for m in methods:
        v = r[m]
        per_sid_rows.append({
            'station': sid, 'method': m, 'seed': r['seed'],
            'acc': v['acc'], 'rmse': v['rmse'], 'mae': v['mae'], 'r2': v['r2']
        })
per_sid = pd.DataFrame(per_sid_rows)
agg = per_sid.groupby(['station', 'method']).agg(
    acc_mean=('acc', 'mean'), acc_std=('acc', 'std'),
    rmse_mean=('rmse', 'mean')
).round(3)
display(agg)